In [ ]:
%pip install pytesseract
%pip install pymupdf opencv-python matplotlib numpy pandas
%pip install camelot-py

In [ ]:
from pathlib import Path
import pytesseract

tess_path = r"C:\Program Files\Tesseract-OCR\tesseract.exe"

print("exists:", Path(tess_path).exists())

pytesseract.pytesseract.tesseract_cmd = tess_path
print("version:", pytesseract.get_tesseract_version())

TESS_AVAILABLE = True
print("OCR available:", TESS_AVAILABLE)

In [ ]:
%pip install openai
from openai import OpenAI
import os
import json
import re
import base64
from pathlib import Path

def build_client():
    return OpenAI(
        api_key=os.getenv("DASHSCOPE_API_KEY"),
        base_url="https://dashscope.aliyuncs.com/compatible-mode/v1",
    )

def image_to_data_url(image_path):
    ext = Path(image_path).suffix.lower().replace(".", "")
    if ext == "jpg":
        ext = "jpeg"
    with open(image_path, "rb") as f:
        b64 = base64.b64encode(f.read()).decode("utf-8")
    return f"data:image/{ext};base64,{b64}"

def extract_first_json(text):
    if isinstance(text, dict):
        return text
    if text is None:
        raise ValueError("AI returned empty text")

    text = str(text).strip()

    try:
        return json.loads(text)
    except:
        pass

    text2 = re.sub(r"^```(?:json)?", "", text, flags=re.I).strip()
    text2 = re.sub(r"```$", "", text2).strip()
    try:
        return json.loads(text2)
    except:
        pass

    start = text.find("{")
    end = text.rfind("}")
    if start >= 0 and end > start:
        return json.loads(text[start:end+1])

    raise ValueError(f"Cannot parse JSON from AI response: {text[:500]}")

def vl_chat_json(image_path, prompt, model_name="qwen3.6-plus"):
    client = build_client()
    resp = client.chat.completions.create(
        model=model_name,
        messages=[
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": prompt},
                    {
                        "type": "image_url",
                        "image_url": {
                            "url": image_to_data_url(image_path)
                        }
                    }
                ]
            }
        ],
        temperature=0
    )
    raw = resp.choices[0].message.content
    return extract_first_json(raw)

In [ ]:
import inspect
print(inspect.getsource(vl_chat_json))

In [ ]:
# API credentials are provided at runtime by the Streamlit password field.


In [ ]:
import os
import json
from pathlib import Path

import cv2
import numpy as np
import fitz
from openai import OpenAI

In [ ]:
# =========================================================
# STAGE 1: UNIVERSAL PDF PREPROCESSING
# No AI used
#
# Purpose:
# 1. Open PDF
# 2. Extract page-level text
# 3. Extract text blocks with PDF coordinates
# 4. Detect figure legend candidates
# 5. Extract image object candidates
# 6. Extract table candidates
# 7. Run Stage 1 QC
#
# Important:
# - Stage 1 only extracts objective evidence.
# - Stage 1 does NOT interpret ELISpot data.
# - Stage 1 does NOT split panels.
# - Stage 1 does NOT infer N_Mice / ELISpot_Unit / Treatment / Dose / Route.
# - Stage 1 preserves source_doc / source_section / source_page for downstream stages.
# =========================================================


# =========================================================
# A. Imports
# =========================================================

import os
import re
import json
from pathlib import Path

import fitz
import cv2
import numpy as np


# =========================================================
# B. Basic utilities
# =========================================================

# More universal figure legend start pattern.
# It can match:
#   Figure 1
#   Fig. 2
#   Fig 3
#   Figure S1
#   Fig. S2
#   Supplementary Fig. S1
#   Supplementary Figure 2
#   Extended Data Fig. 1
#   Scheme 1
FIGURE_START_RE = re.compile(
    r"""
    ^\s*
    (?P<figure_label_raw>
        (?:
            (?:Supplementary|Supplemental)\s+
        )?
        (?:
            Figure|Fig\.?|Scheme
        )
        \s*\.?\s*
        S?\d+[A-Za-z]?
        |
        Extended\s+Data\s+Fig\.?\s*\d+[A-Za-z]?
    )
    \b
    """,
    re.I | re.X
)


def ensure_dir(path):
    """
    Ensure a directory exists.
    """
    Path(path).mkdir(parents=True, exist_ok=True)


def save_json(obj, path):
    """
    Save Python object as JSON.
    """
    Path(path).parent.mkdir(parents=True, exist_ok=True)

    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)


def clean_text(x):
    """
    Clean text while preserving content.
    """
    if x is None:
        return ""

    x = str(x)
    x = x.replace("\u00ad", "")
    x = x.replace("\ufeff", "")
    x = x.replace("￾", "")
    x = x.replace("−", "-")
    x = re.sub(r"\s+", " ", x).strip()

    return x


def page_to_rgb(page, zoom=2.0):
    """
    Render one PDF page to RGB image.

    Note:
    Current downstream Stage 2 / Stage 3 does not use page size or zoom metadata.
    Therefore Stage 1 saves page.png but does not add unused dimension fields.
    """
    mat = fitz.Matrix(zoom, zoom)
    pix = page.get_pixmap(matrix=mat, alpha=False)

    img = np.frombuffer(
        pix.samples,
        dtype=np.uint8
    ).reshape(pix.height, pix.width, pix.n)

    return img


def extract_figure_label_raw(text):
    """
    Extract raw figure label from a legend start line.

    Example:
        "Fig. 2 LPP-CT26 ..." -> "Fig. 2"
        "Figure S1 ..." -> "Figure S1"
    """
    if text is None:
        return ""

    first_line = str(text).splitlines()[0].strip()

    m = FIGURE_START_RE.match(first_line)

    if not m:
        return ""

    return clean_text(m.group("figure_label_raw"))


# =========================================================
# C. Extract full page text
# =========================================================

def get_page_text(page):
    """
    Extract full page text.
    """
    return page.get_text()


# =========================================================
# D. Extract text blocks with PDF coordinates
# =========================================================

def get_text_blocks(page, source_doc=None, source_section=None, source_page=None):
    """
    Extract text blocks with PDF coordinates.

    Each block keeps:
    - block_index
    - bbox
    - text
    - block_no
    - block_type
    - source_doc
    - source_section
    - source_page
    """
    raw_blocks = page.get_text("blocks", sort=True)
    blocks = []

    for i, item in enumerate(raw_blocks):
        if len(item) < 7:
            continue

        x0, y0, x1, y1, text, block_no, block_type = item[:7]
        text = (text or "").strip()

        if not text:
            continue

        blocks.append({
            "block_index": i,
            "bbox": [x0, y0, x1, y1],
            "text": text,
            "block_no": block_no,
            "block_type": block_type,

            "source_doc": source_doc,
            "source_section": source_section,
            "source_page": source_page
        })

    return blocks


# =========================================================
# E. Detect legend candidates
# =========================================================

def find_legend_start_blocks(blocks):
    """
    Find blocks that look like figure legend starts.
    """
    out = []

    for b in blocks:
        first_line = b["text"].splitlines()[0].strip()

        if FIGURE_START_RE.match(first_line):
            candidate = dict(b)
            candidate["figure_label_raw"] = extract_figure_label_raw(first_line)
            out.append(candidate)

    return out


def build_full_legend_from_start(blocks, start_block_index, max_vertical_gap=40):
    """
    Build a full legend from a legend start block.

    Improvements over old version:
    - Keep included_block_indices
    - Keep end_block_index
    - Keep figure_label_raw
    - Keep source fields
    """
    start_block = None

    for b in blocks:
        if b["block_index"] == start_block_index:
            start_block = b
            break

    if start_block is None:
        return None

    parts = [start_block["text"]]
    included_block_indices = [start_block["block_index"]]

    x0, y0, x1, y1 = start_block["bbox"]
    last_bottom = y1
    end_block_index = start_block["block_index"]

    sorted_blocks = sorted(blocks, key=lambda x: x["block_index"])

    start_pos = None

    for i, b in enumerate(sorted_blocks):
        if b["block_index"] == start_block_index:
            start_pos = i
            break

    if start_pos is None:
        return None

    for j in range(start_pos + 1, len(sorted_blocks)):
        b = sorted_blocks[j]
        txt = b["text"].strip()

        if not txt:
            continue

        first_line = txt.splitlines()[0].strip()

        # Stop when the next figure legend starts.
        if FIGURE_START_RE.match(first_line):
            break

        bx0, by0, bx1, by1 = b["bbox"]
        vertical_gap = by0 - last_bottom

        # Stop if the next block is too far away.
        if vertical_gap > max_vertical_gap:
            break

        parts.append(txt)
        included_block_indices.append(b["block_index"])
        end_block_index = b["block_index"]

        last_bottom = by1

        x0 = min(x0, bx0)
        y0 = min(y0, by0)
        x1 = max(x1, bx1)
        y1 = max(y1, by1)

    full_legend = "\n".join(parts).strip()

    return {
        "legend_bbox_pdf": [x0, y0, x1, y1],
        "full_legend": full_legend,
        "start_block_index": start_block_index,
        "end_block_index": end_block_index,
        "included_block_indices": included_block_indices,
        "figure_label_raw": extract_figure_label_raw(full_legend),

        "source_doc": start_block.get("source_doc"),
        "source_section": start_block.get("source_section"),
        "source_page": start_block.get("source_page")
    }


# =========================================================
# F. Extract image objects from PDF object layer
# =========================================================

def extract_image_objects(page, doc, page_num, out_dir, source_doc=None, source_section=None):
    """
    Extract image object candidates from one PDF page.

    Note:
    These are candidates only.
    Some PDF figures may be vector graphics or multi-object composites.
    The full page image is still saved as fallback by process_one_page_stage1().
    """
    ensure_dir(out_dir)
    image_candidates = []

    seen = set()
    images = page.get_images(full=True)

    for i, img in enumerate(images):
        xref = img[0]

        if xref in seen:
            continue

        seen.add(xref)

        try:
            base = doc.extract_image(xref)
            image_bytes = base["image"]
            ext = base["ext"]
        except Exception:
            continue

        try:
            rects = page.get_image_rects(xref)
        except Exception:
            rects = []

        # If bbox cannot be retrieved, still save image object.
        if not rects:
            img_path = os.path.join(out_dir, f"page{page_num}_img{i+1}.{ext}")

            with open(img_path, "wb") as f:
                f.write(image_bytes)

            image_candidates.append({
                "image_index": i + 1,
                "xref": xref,
                "page_num": page_num,
                "source_doc": source_doc,
                "source_section": source_section,
                "source_page": page_num,
                "bbox": None,
                "image_path": img_path,
                "extraction_method": "pdf_object",
                "bbox_available": False
            })

            continue

        for j, rect in enumerate(rects, start=1):
            img_path = os.path.join(out_dir, f"page{page_num}_img{i+1}_{j}.{ext}")

            with open(img_path, "wb") as f:
                f.write(image_bytes)

            image_candidates.append({
                "image_index": i + 1,
                "xref": xref,
                "page_num": page_num,
                "source_doc": source_doc,
                "source_section": source_section,
                "source_page": page_num,
                "bbox": [rect.x0, rect.y0, rect.x1, rect.y1],
                "image_path": img_path,
                "extraction_method": "pdf_object",
                "bbox_available": True
            })

    return image_candidates


# =========================================================
# G. Extract table candidates
# =========================================================

def try_find_tables(page, page_num, source_doc=None, source_section=None):
    """
    Try to find table candidates on one page.
    """
    tables_info = []

    try:
        tables = page.find_tables()

        for idx, tab in enumerate(tables.tables):
            tables_info.append({
                "table_index": idx,
                "page_num": page_num,
           …77734 tokens truncated…_norm,
            vaccine_hint=vaccine_hint,
            ref_df=ref_df
        )

        if ref_row is None:
            ref_row = {}

        route_axis = infer_route_from_axis(r)
        dose_axis = infer_dose_from_axis(r)
        treatment_axis = infer_treatment_from_axis(r)

        mhc_class = clean_text(ref_row.get("MHC_Class", "Not specified"))
        mhc_alleles = extract_mhc_alleles(mhc_class)

        final_row = {
            "Epitope_ID": first_nonempty(
                ref_row.get("Epitope_ID", "Not specified"),
                epitope_id_norm
            ),
            "Epitope_Sequence": clean_text(ref_row.get("Epitope_Sequence", "Not specified")),
            "Epitope_Gene": clean_text(ref_row.get("Epitope_Gene", "Not specified")),
            "Epitope_Mutation": clean_text(ref_row.get("Epitope_Mutation", "Not specified")),
            "MHC_Class": mhc_class,
            "Vaccine_or_Antigen_Set": first_nonempty(
                ref_row.get("Vaccine_or_Antigen_Set", "Not specified"),
                r.get("Legend_Vaccine_or_Antigen_Set", "Not specified")
            ),
            "X_Axis_Value": clean_text(r.get("X_Axis_Value", "Not specified")),
            "Y_Axis_Value": clean_text(r.get("Y_Axis_Value", "Not specified")),
            "ELISpot_Value": safe_float(r.get("ELISpot_Value", np.nan)),
            "ELISpot_Unit": first_nonempty(
                r.get("ELISpot_Unit", "Not specified"),
                r.get("Legend_ELISpot_Unit", "Not specified")
            ),
            "Readout": first_nonempty(
                r.get("Readout", "Not specified"),
                r.get("Assay_Type", "Not specified")
            ),
            "Sample_or_Tissue": clean_text(r.get("Sample_or_Tissue", "Not specified")),
            "Treatment": first_nonempty(
                treatment_axis,
                r.get("Legend_Treatment", "Not specified"),
                r.get("Treatment", "Not specified")
            ),
            "Dose": first_nonempty(
                dose_axis,
                r.get("Legend_Dose", "Not specified")
            ),
            "Route": first_nonempty(
                route_axis,
                r.get("Legend_Route", "Not specified")
            ),
            "N_Mice": clean_text(r.get("Legend_N_Mice", "Not specified")),
            "Epitope_Source_Model": "Stage5_reference_table" if ref_row else "Stage4_heatmap_axis",
            "Mouse_Tumor_Model": clean_text(r.get("Legend_Mouse_Tumor_Model", "Not specified")),
            "Mouse_Strain": clean_text(r.get("Legend_Mouse_Strain", "Not specified")),
            "Mouse_MHC_I_A": mhc_alleles[0] if len(mhc_alleles) > 0 else "Not specified",
            "Mouse_MHC_I_B": mhc_alleles[1] if len(mhc_alleles) > 1 else "Not specified",
            "Mouse_MHC_I_C": mhc_alleles[2] if len(mhc_alleles) > 2 else "Not specified",
            "Article_Source": build_article_source(r),
            "Data_Source": "Stage4_heatmap_cell_records + Stage3C_legend_context + Stage5_reference_table"
        }

        for c in ELISPOT_MAIN_COLUMNS:
            if c not in final_row:
                final_row[c] = "Not specified"

        rows.append(final_row)

        audit_rows.append({
            "Stage4_Row_Index": idx,
            "Stage4_Row_ID": clean_text(r.get("Stage4_Row_ID", "Not specified")),
            "Stage4_Input_File": clean_text(r.get("Stage4_Input_File", "Not specified")),
            "Figure_Unit_ID": clean_text(r.get("Figure_Unit_ID", "Not specified")),
            "Candidate_ID": clean_text(r.get("Candidate_ID", "Not specified")),
            "Panel_ID": clean_text(r.get("Panel_ID", "Not specified")),
            "Row_Index": clean_text(r.get("Row_Index", "Not specified")),
            "Col_Index": clean_text(r.get("Col_Index", "Not specified")),
            "Derived_Epitope_ID_Normalized": epitope_id_norm,
            "Reference_Matched": ref_row != {},
            "Reference_Match_Note": clean_text(ref_note),
            "X_Axis_Role": clean_text(r.get("X_Axis_Role", "Not specified")),
            "Y_Axis_Role": clean_text(r.get("Y_Axis_Role", "Not specified")),
            "X_Axis_Value": clean_text(r.get("X_Axis_Value", "Not specified")),
            "Y_Axis_Value": clean_text(r.get("Y_Axis_Value", "Not specified")),
            "Legend_Treatment": clean_text(r.get("Legend_Treatment", "Not specified")),
            "Legend_Dose": clean_text(r.get("Legend_Dose", "Not specified")),
            "Legend_Route": clean_text(r.get("Legend_Route", "Not specified")),
            "Final_Epitope_ID": final_row["Epitope_ID"],
            "Final_Epitope_Mutation": final_row["Epitope_Mutation"],
            "Final_Treatment": final_row["Treatment"],
            "Final_Dose": final_row["Dose"],
            "Final_Route": final_row["Route"],
            "Final_Vaccine_or_Antigen_Set": final_row["Vaccine_or_Antigen_Set"]
        })

    final_df = pd.DataFrame(rows)

    for c in ELISPOT_MAIN_COLUMNS:
        if c not in final_df.columns:
            final_df[c] = "Not specified"

    final_df = final_df[ELISPOT_MAIN_COLUMNS].copy()
    audit_df = pd.DataFrame(audit_rows)

    return final_df, audit_df


def build_missing_summary(df):
    rows = []

    for c in df.columns:
        missing_count = df[c].apply(
            lambda x: is_not_specified(x) or (isinstance(x, float) and pd.isna(x))
        ).sum()

        rows.append({
            "Column": c,
            "Missing_or_Not_Specified_Count": int(missing_count),
            "Total_Count": int(len(df)),
            "Missing_Rate": float(missing_count / len(df)) if len(df) else 0.0
        })

    return pd.DataFrame(rows)


def build_duplicate_check(df):
    key_cols = [
        "Epitope_ID",
        "X_Axis_Value",
        "Y_Axis_Value",
        "ELISpot_Value",
        "Article_Source"
    ]

    existing = [c for c in key_cols if c in df.columns]

    if not existing or df.empty:
        return pd.DataFrame()

    dup_mask = df.duplicated(subset=existing, keep=False)

    return df[dup_mask].copy()


def make_qc_summary(final_df, stage4_df, ref_df, audit_df, failed_df):
    missing_summary = build_missing_summary(final_df)
    duplicate_df = build_duplicate_check(final_df)

    if not audit_df.empty and "Reference_Matched" in audit_df.columns:
        matched_count = int(audit_df["Reference_Matched"].astype(bool).sum())
        unmatched_count = int((~audit_df["Reference_Matched"].astype(bool)).sum())
    else:
        matched_count = 0
        unmatched_count = 0

    qc = {
        "stage4_input_records": int(len(stage4_df)),
        "reference_table_rows": int(len(ref_df)),
        "final_elispot_rows": int(len(final_df)),
        "unique_epitope_ids_final": int(final_df["Epitope_ID"].nunique(dropna=False)) if not final_df.empty else 0,
        "reference_matched_rows": matched_count,
        "reference_unmatched_rows": unmatched_count,
        "failed_rows": int(len(failed_df)),
        "duplicate_rows_by_main_key": int(len(duplicate_df)),
        "epitope_mutation_not_specified_count": int(final_df["Epitope_Mutation"].apply(is_not_specified).sum()) if not final_df.empty else 0,
        "records_by_article_source": final_df["Article_Source"].value_counts(dropna=False).to_dict()
            if not final_df.empty and "Article_Source" in final_df.columns else {},
        "records_by_vaccine_or_antigen_set": final_df["Vaccine_or_Antigen_Set"].value_counts(dropna=False).to_dict()
            if not final_df.empty and "Vaccine_or_Antigen_Set" in final_df.columns else {},
        "missing_by_column": {
            r["Column"]: int(r["Missing_or_Not_Specified_Count"])
            for _, r in missing_summary.iterrows()
        }
    }

    return qc, missing_summary, duplicate_df


def find_reference_table_auto(root="stage5_reference_table_outputs_final"):
    root = Path(root)

    if not root.exists():
        return None

    files = list(root.rglob("*.xlsx")) + list(root.rglob("*.csv"))

    if not files:
        return None

    scored = []

    for f in files:
        name = f.name.lower()
        score = 0

        if "table" in name:
            score += 3

        if "s1" in name:
            score += 3

        if "neo" in name or "epitope" in name:
            score += 4

        if "final" in name:
            score += 5

        if "reference" in name:
            score += 2

        if "failed" in name or "qc" in name or "summary" in name:
            score -= 10

        scored.append((score, f))

    scored = sorted(scored, key=lambda x: x[0], reverse=True)

    if scored and scored[0][0] > 0:
        return str(scored[0][1])

    return str(files[0])


def run_stage6_final_elispot_merge(
    stage4_cell_records_paths,
    reference_table_path=None,
    out_root="stage6_final_elispot_outputs"
):
    out_root = Path(out_root)
    ensure_dir(out_root)

    if reference_table_path is None:
        reference_table_path = find_reference_table_auto()

    if reference_table_path is None:
        raise FileNotFoundError("Reference table not found. Set reference_table_path manually.")

    stage4_raw = read_many_stage4_records(stage4_cell_records_paths)

    ref_raw = read_table_robust(
        reference_table_path,
        preferred_sheets=[
            "Table_S1_Neoepitopes",
            "Final_Reference_Table",
            "Reference_Table",
            "Table_S1",
            "Neoepitopes"
        ]
    )

    stage4_df = standardize_stage4_records(stage4_raw)
    ref_df = standardize_reference_table(ref_raw)

    final_df, audit_df = build_final_elispot_rows(stage4_df, ref_df)

    failed_df = audit_df[
        audit_df["Derived_Epitope_ID_Normalized"].eq("Not specified")
        | audit_df["Reference_Matched"].eq(False)
    ].copy()

    qc, missing_summary_df, duplicate_df = make_qc_summary(
        final_df=final_df,
        stage4_df=stage4_df,
        ref_df=ref_df,
        audit_df=audit_df,
        failed_df=failed_df
    )

    final_csv = out_root / "ELISpot_Data.csv"
    final_xlsx = out_root / "ELISpot_Data.xlsx"

    ref_csv = out_root / "Table_S1_Neoepitopes_Standardized.csv"
    ref_xlsx = out_root / "Table_S1_Neoepitopes_Standardized.xlsx"

    audit_csv = out_root / "Stage6_Merge_Audit.csv"
    audit_xlsx = out_root / "Stage6_Merge_Audit.xlsx"

    missing_csv = out_root / "Stage6_Missing_Field_Summary.csv"
    duplicate_csv = out_root / "Stage6_Duplicate_Check.csv"
    failed_csv = out_root / "Stage6_Unmatched_or_Review_Items.csv"

    combined_xlsx = out_root / "Stage6_Final_ELISpot_Output.xlsx"
    qc_json = out_root / "Stage6_QC_Summary.json"

    final_df.to_csv(final_csv, index=False, encoding="utf-8-sig")
    final_df.to_excel(final_xlsx, index=False)

    ref_df.to_csv(ref_csv, index=False, encoding="utf-8-sig")
    ref_df.to_excel(ref_xlsx, index=False)

    audit_df.to_csv(audit_csv, index=False, encoding="utf-8-sig")
    audit_df.to_excel(audit_xlsx, index=False)

    missing_summary_df.to_csv(missing_csv, index=False, encoding="utf-8-sig")
    duplicate_df.to_csv(duplicate_csv, index=False, encoding="utf-8-sig")
    failed_df.to_csv(failed_csv, index=False, encoding="utf-8-sig")

    save_json(qc, qc_json)

    with pd.ExcelWriter(combined_xlsx, engine="openpyxl") as writer:
        final_df.to_excel(writer, sheet_name="ELISpot_Data", index=False)
        ref_df.to_excel(writer, sheet_name="Table_S1_Neoepitopes", index=False)
        audit_df.to_excel(writer, sheet_name="Stage6_Merge_Audit", index=False)
        missing_summary_df.to_excel(writer, sheet_name="Missing_Field_Summary", index=False)
        duplicate_df.to_excel(writer, sheet_name="Duplicate_Check", index=False)
        failed_df.to_excel(writer, sheet_name="Unmatched_or_Review", index=False)

        pd.DataFrame([
            {
                "Metric": k,
                "Value": json.dumps(v, ensure_ascii=False) if isinstance(v, (dict, list)) else v
            }
            for k, v in qc.items()
        ]).to_excel(writer, sheet_name="QC_Summary", index=False)

    print("=" * 120)
    print("Stage 6 complete.")
    print("Stage 4 inputs:", stage4_cell_records_paths)
    print("Reference input:", reference_table_path)
    print("Final ELISpot rows:", len(final_df))
    print("Reference rows:", len(ref_df))
    print("Reference matched rows:", qc["reference_matched_rows"])
    print("Reference unmatched rows:", qc["reference_unmatched_rows"])
    print("Epitope_Mutation Not specified:", qc["epitope_mutation_not_specified_count"])
    print("Duplicate rows by main key:", qc["duplicate_rows_by_main_key"])
    print("Final Excel:", combined_xlsx)
    print("ELISpot_Data:", final_xlsx)
    print("QC:", qc_json)

    display(final_df.head(50))
    display(ref_df.head(20))
    display(missing_summary_df)
    display(failed_df.head(50))

    return {
        "elispot_data_df": final_df,
        "reference_df": ref_df,
        "audit_df": audit_df,
        "missing_summary_df": missing_summary_df,
        "duplicate_df": duplicate_df,
        "failed_df": failed_df,
        "qc": qc,
        "elispot_data_csv": final_csv,
        "elispot_data_xlsx": final_xlsx,
        "reference_csv": ref_csv,
        "reference_xlsx": ref_xlsx,
        "audit_csv": audit_csv,
        "audit_xlsx": audit_xlsx,
        "missing_csv": missing_csv,
        "duplicate_csv": duplicate_csv,
        "failed_csv": failed_csv,
        "combined_xlsx": combined_xlsx,
        "qc_json": qc_json
    }


STAGE4_CELL_RECORDS_PATHS = [
    "stage4_final_generic_outputs_after_stage3b3c/Stage4_Cell_Records.xlsx"
]

REFERENCE_TABLE_PATH = None

stage6_result = run_stage6_final_elispot_merge(
    stage4_cell_records_paths=STAGE4_CELL_RECORDS_PATHS,
    reference_table_path=REFERENCE_TABLE_PATH,
    out_root="stage6_final_elispot_outputs"
)